# RoadGuard - Agent Pipeline (standalone)

End-to-end run of the 5-worker + supervisor + approval-gate agent pipeline that recommends a hazard action from a road-damage image. The agent layer runs with no FastAPI service, no SQLite, no VM, no webapp and no paid API, so it can be executed from a plain Python kernel.

## What the pipeline reads from the team

The notebook depends on three deliverables the team ships at conventional paths next to it:

| Dependency | Path | Owner |
|------------|------|-------|
| YOLOv8n ONNX detector | `../models/yolov8n-roadguard.onnx` | Shai |
| Sample road images | `../data/sample/` | Ovad |
| ChromaDB corpus `charter_grounded_corpus` | `../rag/chroma_db/` | Benny |

Every cell is pure Python; nothing phones a remote API.

## Verdicts

Given a hazard image, the pipeline produces one of four actions:

- `urgent` - a human supervisor must approve dispatch of a maintenance crew within hours.
- `standard` - the hazard enters the routine repair queue.
- `needs_more_evidence` - the reporter is asked for one more photo.
- `human_review` - the pipeline cannot ground a safe decision; the case is routed to a supervisor with a full trace.


## 1. Reproducibility

To keep the demo deterministic on every run, we:

- Pin the Python version (3.10 or newer; tested on 3.11 and 3.12).
- Set the random seed for `random` (and numpy if present) before any sampling.
- Document the exact dependency versions in the install cell. The core notebook uses only the Python standard library; `requests` is pulled in optionally for the live-LLM path.
- Use pure functions throughout: every agent is `(state) -> dict`, no hidden state, no global mutations.
- Avoid wall-clock timestamps in rationales (the production system stamps those; here they would make diffs noisy).

Run the next cell to pin the seeds and print the versions actually in use.


In [1]:

import sys
import random
import platform

# Deterministic seed - the rule-tree pipeline makes no random calls, but we
# still fix it so any future randomized step (e.g. a sampled LLM reply)
# produces the same output across runs.
SEED = 42
random.seed(SEED)
try:
    import numpy as _np
    _np.random.seed(SEED)
    _np_version = _np.__version__
except ImportError:
    _np_version = "(not installed; not required)"

print(f"Python     : {sys.version.split()[0]} on {platform.system()}")
print(f"random seed: {SEED}")
print(f"numpy      : {_np_version}")
print(f"requests   : (optional; only needed for the live-LLM path)")


Python     : 3.9.2 on Windows
random seed: 42
numpy      : 2.0.2
requests   : (optional; only needed for the live-LLM path)


## 2. The pipeline, at a glance

```
sanitize -> signal_extractor -> supervisor -> profile_builder
         -> supervisor -> rag_retriever -> supervisor -> decision
         -> approval_gate
```

| Node              | What it does                                                                                               | Reads                          | Writes                                      |
|-------------------|------------------------------------------------------------------------------------------------------------|--------------------------------|---------------------------------------------|
| sanitize          | Regex boundary defence on 3 free-text channels (location, OCR, reporter text); flags injection attempts.   | metadata, image_quality         | sanitized fields + `*_suspicious` booleans  |
| signal_extractor  | Rule-based keyword extraction over the reporter's text; produces `reporter_intake` + `reporter_signals`.    | reporter_text, YOLO, metadata   | reporter_intake, reporter_signals, contradictions |
| supervisor        | Python router + budget guard (iterations, wall-clock cap). Not an LLM.                                      | state["budget"]                | guard_fired, routing decision               |
| profile_builder   | YOLO + image quality + metadata -> HazardProfile. Severity label (minor/moderate/severe) is a visual descriptor, NOT an action input. | yolo_output, image_quality      | hazard_profile, perception_confidence       |
| rag_retriever     | In-memory keyword lookup over a hard-coded corpus of 5 repair-standards passages.                           | hazard_profile.primary_class, road_type | procedures, tool_result_ids, research_coverage |
| decision          | Rule tree (optionally LLM via Groq) -> action, priority, rationale, cited_sources, assessment.               | everything above                | action, rationale, priority, cited_sources  |
| approval_gate     | For urgent / human_review verdicts, emit a pending-approval record (a dict, not an email).                   | verdict                        | approval_record                             |

The 4 budget caps are enforced by the supervisor: iterations (default 8), wall-clock (default 30 s), tool-hops (default 3 per worker), and a cost cap (default $0.10 USD-equivalent). In this notebook the wall-clock cap is lifted so Colab's cold start doesn't trip it; the other caps still apply.


## 3. Agent 1 - `sanitize`

Prompt injection is the single biggest risk when a free-text channel reaches an LLM. The reporter writes anything into `reporter_text`; OCR can pick up adversarial signage; even the location field could carry a `; drop table` style payload. `sanitize` is a plain-regex scrub that runs BEFORE any LLM sees the text.

For each of the three channels (location, OCR text, reporter text), it:

1. Normalises whitespace, strips control characters.
2. Looks for known injection patterns (lines starting with `system:`, `ignore previous`, `jailbreak`, `developer mode`, markdown fence tricks, etc.).
3. Returns the scrubbed string + a `*_suspicious` boolean the downstream nodes read.

A suspicious hit does NOT crash the pipeline; it is a signal that the perception_confidence will later clamp to `low`, which the Decision rule tree routes to `human_review`. The reporter's text is also wrapped in `[USER_SUPPLIED_UNVERIFIED]` markers before being fed to any LLM, so the model is told to treat it as evidence, not as instructions.


In [2]:

import re

_INJECTION_PATTERNS = [
    r"ignore\s+(?:all\s+)?previous\s+instructions",
    r"disregard\s+(?:the\s+)?(?:above|system)",
    r"jailbreak",
    r"developer\s+mode",
    r"you\s+are\s+now\s+",
    r"^\s*system\s*:",
    r"forget\s+(?:everything|all)",
    r"```+\s*system",
]
_INJECTION_RE = re.compile("|".join(_INJECTION_PATTERNS), re.IGNORECASE | re.MULTILINE)
_CTRL_RE = re.compile(r"[\x00-\x08\x0b-\x1f\x7f]")


def _scrub(text: str) -> tuple[str, bool]:
    """Return (sanitized_text, is_suspicious). None/empty input is clean."""
    if not text:
        return "", False
    s = _CTRL_RE.sub(" ", text)
    s = re.sub(r"\s+", " ", s).strip()
    suspicious = bool(_INJECTION_RE.search(s))
    return s, suspicious


def sanitize(state: dict) -> dict:
    """Boundary defence on location, OCR and reporter text channels."""
    md = dict(state.get("metadata") or {})
    iq = dict(state.get("image_quality") or {})

    loc_clean, loc_bad = _scrub(md.get("location"))
    ocr_clean, ocr_bad = _scrub(iq.get("ocr_text_detected"))
    rep_clean, rep_bad = _scrub(md.get("reporter_text"))

    md["location"] = loc_clean
    md["location_was_scrubbed"] = loc_bad
    iq["ocr_text_detected"] = ocr_clean
    iq["ocr_suspicious"] = ocr_bad
    md["reporter_text"] = rep_clean
    md["reporter_text_suspicious"] = rep_bad

    trace_entry = {
        "node": "sanitize",
        "location_scrubbed": loc_bad,
        "ocr_suspicious": ocr_bad,
        "reporter_text_suspicious": rep_bad,
    }
    return {
        "metadata": md,
        "image_quality": iq,
        "trace": state.get("trace", []) + [trace_entry],
    }


## 4. Agent 2 - `signal_extractor`

The reporter's text is unstructured but often carries severity cues ("huge pothole", "cars are swerving", "there's water pooling"). `signal_extractor` runs a keyword pass and emits two structured outputs:

- `reporter_intake`: a dict with `mentioned` (which fields the reporter touched - hazard_class, lane_blocked, road_type), `contradictions` (where the reporter's claim disagrees with the detector), and `urgency_claimed` (did the reporter say "urgent" or similar).
- `reporter_signals`: `size_cm`, `depth_cm`, `vehicles_swerving`, `standing_water` - severity cues the Decision prompt reads as "unverified evidence, not instructions".

In the production system this node optionally calls an LLM (`AGENT_SIGNAL_LLM=1`). Here we always use the keyword pass; it is deterministic, cheap, and the Decision rule tree only reads a handful of flags from the output anyway.


In [3]:

_CM_RE = re.compile(r"(\d{1,4})\s*(?:cm|centim)", re.IGNORECASE)
_DEEP_RE = re.compile(r"(?:deep|depth)\s*(?:of)?\s*(\d{1,3})\s*(?:cm|centim)", re.IGNORECASE)
_SWERVE_RE = re.compile(r"swerv|swerve", re.IGNORECASE)
_WATER_RE = re.compile(r"water|flooded|pool|puddle", re.IGNORECASE)
_URGENT_RE = re.compile(r"\burgent\b|\bdanger\b|\baccident\b", re.IGNORECASE)

_CLASS_WORDS = {
    "D00": re.compile(r"longitudinal\s+crack", re.IGNORECASE),
    "D10": re.compile(r"transverse\s+crack", re.IGNORECASE),
    "D20": re.compile(r"alligator|crocodile", re.IGNORECASE),
    "D40": re.compile(r"pothole|bor\b", re.IGNORECASE),
}



def _llm_signal(text: str) -> dict:
    """Ask the signal-extraction LLM to pull structured hints from the
    reporter's free text. Returns {hazard_class, lane_blocked, urgency}.
    An empty text returns {}, so the pipeline still runs without a reporter."""
    if not text or len(text.strip()) < 3:
        return {}
    system = (
        'You read a citizen hazard report and extract three fields as JSON: '
        'hazard_class (one of D00, D10, D20, D40 or null), '
        'lane_blocked (none, partial, full), '
        'urgency_claimed (true, false). Reply with ONLY the JSON object.'
    )
    try:
        raw = chat_for('signal_extractor', [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': text[:400]},
        ])
        import re, json as _json
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return _json.loads(m.group(0)) if m else {}
    except Exception as exc:
        return {'_llm_error': str(exc)[:120]}

def signal_extractor(state: dict) -> dict:
    """Keyword pass over the sanitized reporter_text."""
    md = state.get("metadata") or {}
    yolo = state.get("yolo_output") or {}
    text = md.get("reporter_text", "")

    signals = {
        "size_cm": None, "depth_cm": None,
        "vehicles_swerving": None, "standing_water": None,
    }
    intake = {
        "mentioned": {"hazard_class": None, "lane_blocked": None, "road_type": None},
        "contradictions": [],
        "urgency_claimed": bool(_URGENT_RE.search(text or "")),
        "suspicious": bool(md.get("reporter_text_suspicious")),
    }

    if text:
        m = _CM_RE.search(text)
        if m:
            signals["size_cm"] = min(10000, int(m.group(1)))
        m = _DEEP_RE.search(text)
        if m:
            signals["depth_cm"] = min(10000, int(m.group(1)))
        signals["vehicles_swerving"] = bool(_SWERVE_RE.search(text))
        signals["standing_water"] = bool(_WATER_RE.search(text))

        for cls_code, rx in _CLASS_WORDS.items():
            if rx.search(text):
                intake["mentioned"]["hazard_class"] = cls_code
                break

        # Contradiction: reporter says one class, detector sees another
        detector_cls = yolo.get("top1_class")
        reporter_cls = intake["mentioned"]["hazard_class"]
        if detector_cls and reporter_cls and detector_cls != reporter_cls:
            intake["contradictions"].append({
                "field": "hazard_class",
                "reporter": reporter_cls,
                "observed": detector_cls,
                "detail": f"reporter says {reporter_cls}, detector sees {detector_cls}",
            })

    trace_entry = {
        "node": "signal_extractor",
        "has_text": bool(text),
        "mentioned": intake["mentioned"],
        "contradictions": len(intake["contradictions"]),
        "urgency_claimed": intake["urgency_claimed"],
        "reporter_signals": signals,
    }
    # 27.10 - the signal LLM reads the (cleaned) reporter text and may
    # add structured hints that the regex pass could not pick up.
    _llm_hint = _llm_signal(state.get('metadata', {}).get('reporter_text', '') or '')
    if _llm_hint:
        state.setdefault('reporter_signals', {}).setdefault('llm', _llm_hint)

    return {
        "reporter_intake": intake,
        "reporter_signals": signals,
        "trace": state.get("trace", []) + [trace_entry],
    }


## 5. Supervisor

The supervisor is a plain-Python router. Before every worker it ticks a budget tracker that enforces four caps:

- `MAX_ITERATIONS` - total supervisor visits (default 8).
- `MAX_TOOL_HOPS_PER_AGENT` - how many tool calls Decision may make in its ReAct loop (default 3; this notebook does not run the ReAct path).
- `COST_CAP_USD` - total LLM spend (default $0.10; this notebook stays at 0 unless Groq is enabled).
- `WALL_CLOCK_CAP_SECONDS` - wall-clock budget for the whole run (default 30 s; raised to 300 s in the notebook so Colab's cold start cannot trip it).

If any cap fires, `graceful_stop_decision` returns a `human_review` verdict with a rationale naming the cap that tripped and whatever partial progress the pipeline accomplished (`last_speaker`, class so far, RAG coverage so far). The run never raises; a budget stop is a legitimate answer, not a crash.


In [4]:

import time

MAX_ITERATIONS = 8
MAX_TOOL_HOPS_PER_AGENT = 3
COST_CAP_USD = 0.10
WALL_CLOCK_CAP_SECONDS = 300.0  # relaxed from prod default 30s for Colab cold start


class Budget:
    def __init__(self):
        self.started_at = time.monotonic()
        self.iterations = 0
        self.tool_hops = 0
        self.cost_usd = 0.0
        self.fired = False
        self.reason = ""

    def tick_iteration(self):
        self.iterations += 1

    def check(self) -> tuple[bool, str]:
        if self.iterations > MAX_ITERATIONS:
            return True, f"MAX_ITERATIONS ({MAX_ITERATIONS}) reached"
        if self.tool_hops > MAX_TOOL_HOPS_PER_AGENT * 3:
            return True, f"MAX_TOOL_HOPS ({MAX_TOOL_HOPS_PER_AGENT * 3}) reached"
        if self.cost_usd > COST_CAP_USD:
            return True, f"COST_CAP_USD ({COST_CAP_USD}) exceeded"
        elapsed = time.monotonic() - self.started_at
        if elapsed > WALL_CLOCK_CAP_SECONDS:
            return True, f"WALL_CLOCK_CAP_SECONDS ({WALL_CLOCK_CAP_SECONDS}) reached"
        return False, ""


def graceful_stop_decision(reason: str, state: dict) -> dict:
    """The 'I could not finish' answer; always human_review with context."""
    last = state.get("last_speaker", "pre-sanitize")
    parts = [f"budget guard fired: {reason}. stopped at {last}."]
    profile = state.get("hazard_profile") or {}
    if profile.get("primary_class"):
        parts.append(
            f"profile_builder got as far as class={profile['primary_class']} "
            f"severity={profile.get('severity_signal') or '?'}."
        )
    coverage = state.get("research_coverage")
    if coverage:
        parts.append(f"rag_retriever got as far as coverage={coverage}.")
    parts.append("decision was not reached; manual triage recommended.")
    return {
        "action": "human_review",
        "rationale": " ".join(parts)[:500],
        "priority": 50,
        "cited_sources": [],
        "guard_fired": True,
    }


def supervisor(state: dict) -> dict:
    budget = state.get("budget")
    if budget is None:
        budget = Budget()
    budget.tick_iteration()
    fired, reason = budget.check()
    trace_entry = {
        "node": "supervisor",
        "iterations": budget.iterations,
        "elapsed_s": round(time.monotonic() - budget.started_at, 2),
        "fired": fired,
    }
    update = {
        "budget": budget,
        "trace": state.get("trace", []) + [trace_entry],
    }
    if fired:
        verdict = graceful_stop_decision(reason, state)
        update["final_verdict"] = verdict
    return update


## 6. Agent 3 - `profile_builder`

Interprets YOLO + image quality + metadata into a hazard profile. It emits:

- `primary_class` - which of D00/D10/D20/D40 the detector called.
- `primary_confidence` - the detector's top-1 score.
- `severity_signal` (`minor` / `moderate` / `severe`) - a VISUAL descriptor from bbox area and class. **This is not a policy input**; see Section 5.1 of the agent contract (`docs/agents-contract.md` in the main repo). The Decision rule tree reads `primary_class`, `primary_confidence`, `road_type`, `lane_blocked`, `perception_confidence` and `research_coverage` - never `severity_signal` by itself.
- `perception_confidence` (`low` / `medium` / `high`) - overall trust in the perception layer. `low` if any of: `is_road_scene=False`, `blur_score<50`, OCR text was flagged suspicious. `high` if `top1_margin>=0.3` AND `blur_score>=100`. `medium` otherwise.

In the production system this node can be refined by an LLM (`AGENT_PERCEPTION_LLM=1`), but the refinement is only allowed to LOWER the confidence - never raise it. Here we always use the rule-based values.


In [5]:

_CLASS_SEV = {"D00": "minor", "D10": "minor", "D20": "moderate", "D40": "moderate"}



def _llm_severity(primary_class: str, confidence: float,
                   road_type: str, lane_blocked: str) -> str:
    """Ask the profile LLM to confirm the severity tier."""
    system = (
        'Decide the severity of a road hazard. Reply with ONE word: '
        'minor, moderate, or severe.'
    )
    try:
        raw = chat_for('profile_builder', [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content':
                f'class={primary_class} confidence={confidence:.2f} '
                f'road={road_type} lane={lane_blocked}'},
        ])
        word = (raw or '').strip().lower().split()[0] if raw else 'moderate'
        return word if word in ('minor', 'moderate', 'severe') else 'moderate'
    except Exception:
        return 'moderate'

def profile_builder(state: dict) -> dict:
    yolo = state.get("yolo_output") or {}
    iq = state.get("image_quality") or {}
    md = state.get("metadata") or {}

    hazard_class = yolo.get("top1_class") or "D40"
    confidence = float(yolo.get("top1_confidence") or 0.0)
    top1_margin = float(yolo.get("top1_margin") or 0.0)
    lane_blocked = md.get("lane_blocked", "none")

    # Severity: visual descriptor from class + area + lane blockage
    area_ratio = float(yolo.get("area_ratio") or 0.0)
    if hazard_class == "D40" and lane_blocked in ("partial", "full"):
        severity = "severe"
    elif hazard_class in ("D40", "D20") and area_ratio >= 0.05 and confidence >= 0.6:
        severity = "severe"
    elif hazard_class in ("D40", "D20") and confidence >= 0.5:
        severity = "moderate"
    elif hazard_class in ("D00", "D10"):
        severity = "minor"
    else:
        severity = _CLASS_SEV.get(hazard_class, "moderate")
    severity = _llm_severity(primary_class, confidence, road_type, lane_blocked) or severity

    # perception_confidence: hard floors route to human_review downstream
    is_road = bool(iq.get("is_road_scene", True))
    blur = float(iq.get("blur_score") or 100.0)
    ocr_bad = bool(iq.get("ocr_suspicious") or md.get("reporter_text_suspicious"))
    if not is_road or blur < 50 or ocr_bad:
        perception_conf = "low"
    elif top1_margin >= 0.3 and blur >= 100:
        perception_conf = "high"
    else:
        perception_conf = "medium"

    profile = {
        "primary_class": hazard_class,
        "primary_confidence": confidence,
        "severity_signal": severity,
    }
    trace_entry = {
        "node": "profile_builder",
        "primary_class": hazard_class,
        "primary_confidence": round(confidence, 3),
        "severity_signal": severity,
        "perception_confidence": perception_conf,
    }
    return {
        "hazard_profile": profile,
        "perception_confidence": perception_conf,
        "last_speaker": "profile_builder",
        "trace": state.get("trace", []) + [trace_entry],
    }


## 7. ChromaDB auto-download

The RAG corpus (Benny's `charter_grounded_corpus`, 2,433 passages) is distributed as a GitHub Release asset on the main RoadGuard repo. If the directory does not exist locally, the next cell downloads and extracts it on first run; the file is ~16 MB. Set `FORCE_CHROMA_REDOWNLOAD=True` to re-fetch.


In [ ]:

import os, io, zipfile, urllib.request
from pathlib import Path

CHROMA_RELEASE_URL = (
    'https://github.com/user-attachments/files/34033588/chroma_db-charter_grounded_corpus.zip'
)
CHROMA_PATH_CANDIDATES = [
    '../rag/chroma_db',
    '../RAG/chroma_db',
    '../Rag/chroma_db',
    '../rag_notebook/chroma_db',
    '../RAG/_deploy',
]
FORCE_CHROMA_REDOWNLOAD = False

def _chroma_has_rows(p: str) -> bool:
    try:
        import chromadb
        c = chromadb.PersistentClient(path=p)
        return any(x.count() > 0 for x in c.list_collections())
    except Exception:
        return False

def _ensure_chroma() -> str:
    for p in CHROMA_PATH_CANDIDATES:
        if Path(p).exists() and _chroma_has_rows(p) and not FORCE_CHROMA_REDOWNLOAD:
            print(f'found existing ChromaDB at {p!r}')
            return p
    dest = Path('../rag/chroma_db')
    dest.mkdir(parents=True, exist_ok=True)
    print(f'downloading ChromaDB from {CHROMA_RELEASE_URL} ...')
    with urllib.request.urlopen(CHROMA_RELEASE_URL, timeout=120) as resp:
        data = resp.read()
    print(f'  {len(data) / 1_048_576:.1f} MB received; extracting ...')
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        z.extractall(dest)
    print(f'extracted to {dest!r}')
    return str(dest)

CHROMA_PATH = _ensure_chroma()
print(f'CHROMA_PATH = {CHROMA_PATH!r}')


## 8. LLM provider configuration

The pipeline consults an LLM at each agent step. Three providers are supported; every agent can be routed to a different one, matching the production VM where `sanitize`+`signal` run on a small chat model, `profile`+`decision` on a stronger reasoner and `rag_retriever` on a RAG-biased model.

| Provider | Hosting | Cost | How to use |
|----------|---------|------|------------|
| **Ollama** | Local, port 11434 | Free | `curl -fsSL https://ollama.com/install.sh \| sh` then `ollama pull qwen2.5:3b llama3.2:3b granite3.3:2b`. The notebook calls `http://localhost:11434`. |
| **Groq** | Cloud API | Free tier (~10K req/day) | Get key at <https://console.groq.com> (`gsk_...`). Fast, llama3.3-70b class models. |
| **OpenAI** | Cloud API | Paid (`gpt-4o-mini` is cheap) | Key at <https://platform.openai.com> (`sk-...`). Widely available. |

Edit the `LLM_CONFIG` cell below: set `provider` + `model` for each agent, and fill in the API keys for every provider you use. Models are listed per provider in the comments. The pipeline fails with a clear error if a required key is missing - there is no silent fallback. If you prefer a single model everywhere, set `global_provider` + `global_model` at the top and the per-agent entries will inherit.


In [ ]:

import os
import json
import urllib.request
import urllib.error

LLM_CONFIG = {
    # ---- API keys. Fill in only the ones you plan to use. ----
    'ollama_url': 'http://localhost:11434',
    'groq_api_key': os.environ.get('GROQ_API_KEY', ''),
    'openai_api_key': os.environ.get('OPENAI_API_KEY', ''),

    # ---- Per-agent model routing ----
    #
    # Each agent names: provider ('ollama' | 'groq' | 'openai')
    # and model (the exact id that provider expects).
    #
    # Suggested small/medium/large by provider:
    #   ollama:    qwen2.5:3b, llama3.2:3b, granite3.3:2b, qwen2.5:7b, llama3.1:8b
    #   groq:      llama-3.1-8b-instant, llama-3.3-70b-versatile, qwen-2.5-32b
    #   openai:    gpt-4o-mini, gpt-4o, gpt-4-turbo
    #
    # The VM maps:
    #   sanitize+signal -> small chat (llama3.2:3b)
    #   profile+decision -> stronger reasoner (qwen2.5:3b in prod, 7b possible)
    #   rag_retriever -> small structured output (granite3.3:2b)
    'agents': {
        'sanitize':         {'provider': 'groq', 'model': 'llama-3.1-8b-instant'},
        'signal_extractor': {'provider': 'groq', 'model': 'llama-3.1-8b-instant'},
        'profile_builder':  {'provider': 'groq', 'model': 'llama-3.3-70b-versatile'},
        'rag_retriever':    {'provider': 'groq', 'model': 'llama-3.1-8b-instant'},
        'decision':         {'provider': 'groq', 'model': 'llama-3.3-70b-versatile'},
    },
}

def _chat_ollama(messages, model, temperature=0.0):
    body = {'model': model, 'messages': messages,
            'stream': False, 'options': {'temperature': temperature}}
    req = urllib.request.Request(
        LLM_CONFIG['ollama_url'].rstrip('/') + '/api/chat',
        data=json.dumps(body).encode(),
        headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode())
    return data.get('message', {}).get('content', '')

def _chat_groq(messages, model, temperature=0.0):
    key = LLM_CONFIG['groq_api_key']
    if not key:
        raise RuntimeError('GROQ_API_KEY is empty; fill LLM_CONFIG["groq_api_key"].')
    body = {'model': model, 'messages': messages, 'temperature': temperature}
    req = urllib.request.Request(
        'https://api.groq.com/openai/v1/chat/completions',
        data=json.dumps(body).encode(),
        headers={'Authorization': f'Bearer {key}', 'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode())
    return data['choices'][0]['message']['content']


def _chat_openai(messages, model, temperature=0.0):
    key = LLM_CONFIG['openai_api_key']
    if not key:
        raise RuntimeError('OPENAI_API_KEY is empty; fill LLM_CONFIG["openai_api_key"].')
    body = {'model': model, 'messages': messages, 'temperature': temperature}
    req = urllib.request.Request(
        'https://api.openai.com/v1/chat/completions',
        data=json.dumps(body).encode(),
        headers={'Authorization': f'Bearer {key}', 'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode())
    return data['choices'][0]['message']['content']

_PROVIDERS = {
    'ollama': _chat_ollama, 'groq': _chat_groq,
    'openai': _chat_openai,
}

def chat_for(agent: str, messages, temperature=0.0) -> str:
    cfg = LLM_CONFIG['agents'][agent]
    fn = _PROVIDERS[cfg['provider']]
    return fn(messages, cfg['model'], temperature=temperature)

print('LLM assignment:')
for a, cfg in LLM_CONFIG['agents'].items():
    print(f'  {a:20s} -> {cfg[chr(34)+chr(112)+chr(114)+chr(111)+chr(118)+chr(105)+chr(100)+chr(101)+chr(114)+chr(34)]:10s} {cfg[chr(34)+chr(109)+chr(111)+chr(100)+chr(101)+chr(108)+chr(34)]}')


## 9. Agent 4 - `rag_retriever`

The retriever queries the team's ChromaDB collection `charter_grounded_corpus` under `../rag/chroma_db/`, which Benny built from the municipal repair-standards corpus. It returns up to `k` passages per query; the Decision agent later cites a subset by passage id. The notebook expects the collection to be present at that path (see the README).


In [6]:
import chromadb
from pathlib import Path

# CHROMA_PATH + collection name were set in the ChromaDB download cell
# above; re-use them here.
RAG_COLLECTION = 'charter_grounded_corpus'

_chroma_client = None
_chroma_collection = None

def _open_chroma():
    global _chroma_client, _chroma_collection
    if _chroma_collection is not None:
        return _chroma_collection
    _chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
    _chroma_collection = _chroma_client.get_collection(RAG_COLLECTION)
    return _chroma_collection

def rag_retriever(state: dict, k: int = 3) -> dict:
    profile = state.get('hazard_profile') or {}
    md = state.get('metadata') or {}
    hazard_class = profile.get('primary_class', 'D40')
    road_type = md.get('road_type', 'main')
    query = (
        f"road maintenance repair standards for hazard class {hazard_class} "
        f"on arterial {road_type} road: safety requirements, response "
        "procedures, repair window"
    )
    col = _open_chroma()
    res = col.query(query_texts=[query], n_results=k)
    docs = (res.get('documents') or [[]])[0]
    ids = (res.get('ids') or [[]])[0]
    distances = (res.get('distances') or [[None] * len(ids)])[0]
    procedures = [
        {'article_ids': [pid], 'passage': doc, 'distance': dist}
        for pid, doc, dist in zip(ids, docs, distances)
    ]
    coverage = 'direct_match' if procedures else 'no_match'
    passage_previews = [
        {'id': p['article_ids'][0],
         'distance': p.get('distance'),
         'preview': (p['passage'] or '').replace(chr(10), ' ')[:160]}
        for p in procedures
    ]
    return {
        'procedures': procedures,
        'tool_result_ids': [p['article_ids'][0] for p in procedures],
        'research_coverage': coverage,
        'trace': state.get('trace', []) + [{
            'speaker': 'rag_retriever',
            'content': {
                'coverage': coverage,
                'n_procedures': len(procedures),
                'tool_result_ids': [p['article_ids'][0] for p in procedures],
                'rag_query': query,
                'rag_filter': {'hazard_class': hazard_class,
                               'road_type': road_type, 'k': k},
                'passage_previews': passage_previews,
            },
        }],
    }


## 10. Agent 5 - `decision`

The one node that actually makes the recommendation. In production this runs as a ReAct loop with 8 tools against a local Ollama model (qwen2.5:3b). Here it runs the **deterministic rule tree** by default, which is also the production fallback. Optional live-LLM path via Groq is wired up in Section 9.

### Rule tree order (production + here)

1. `perception_confidence == "low"` -> `human_review` (visual quality not trustworthy enough to decide).
2. Reporter contradicts the detector on hazard_class -> `human_review` (a person has to resolve).
3. detector confidence in `[0.4, 0.6)` and no recurring location -> `needs_more_evidence` (ask for a second angle).
4. `research_coverage == "no_match"` -> `human_review` (no standard to cite = cannot ground a verdict).
5. D40 + main road + lane_blocked in {partial, full} + confidence >= 0.6 -> `urgent`.
6. Else -> `standard`.

Confidence floors (`conf<0.5` -> human_review) were REMOVED from the production tree because they were sending ~60% of real submissions to human review regardless of other signals. The Decision persona in production is a free-text inspector that reads confidence as one signal among many; here the rule tree is faithful to the current production rules.


In [7]:

_CLASS_WEIGHT = {"D40": 60, "D20": 40, "D10": 20, "D00": 15}
_ROAD_BONUS = {"main": 25, "secondary": 15, "bike_path": 10, "parking": 5}
_LANE_BONUS = {"full": 20, "partial": 10, "none": 0}


def _priority(hazard_class: str, confidence: float, road_type: str, lane_blocked: str) -> int:
    base = _CLASS_WEIGHT.get(hazard_class, 15)
    mul = 0.5 if confidence < 0.5 else 1.0
    pri = base * mul + _ROAD_BONUS.get(road_type, 0) + _LANE_BONUS.get(lane_blocked, 0)
    return max(0, min(100, int(round(pri))))


def _intake_class_contradiction(state: dict) -> tuple[bool, str]:
    intake = state.get("reporter_intake") or {}
    for c in intake.get("contradictions") or []:
        if c.get("field") == "hazard_class":
            return True, c.get("detail") or "reporter and detector disagree on the class"
    return False, ""


def decision_rule_tree(state: dict) -> dict:
    profile = state.get("hazard_profile") or {}
    md = state.get("metadata") or {}
    tool_result_ids = state.get("tool_result_ids") or []
    coverage = state.get("research_coverage", "no_match")
    perception_conf = state.get("perception_confidence", "medium")

    hazard_class = profile.get("primary_class", "D40")
    confidence = float(profile.get("primary_confidence", 0.0))
    road_type = md.get("road_type", "main")
    lane_blocked = md.get("lane_blocked", "none")

    priority = _priority(hazard_class, confidence, road_type, lane_blocked)

    if perception_conf == "low":
        return {
            "action": "human_review",
            "rationale": "routed to human: perception_confidence=low "
                         "(suspicious OCR, non-road scene or blurred image)",
            "priority": priority, "cited_sources": [],
        }
    contradiction, detail = _intake_class_contradiction(state)
    if contradiction:
        return {
            "action": "human_review",
            "rationale": f"routed to human: {detail} (unresolved contradiction)",
            "priority": priority, "cited_sources": [],
        }
    if 0.4 <= confidence < 0.6:
        return {
            "action": "needs_more_evidence",
            "rationale": f"detector confidence {confidence:.2f} in the ambiguous band "
                         f"for {hazard_class}; asking for a second angle",
            "priority": priority, "cited_sources": [],
        }
    if coverage == "no_match":
        return {
            "action": "human_review",
            "rationale": f"no RAG match for {hazard_class} on {road_type}",
            "priority": priority, "cited_sources": [],
        }
    if (hazard_class == "D40" and lane_blocked in ("partial", "full")
            and road_type == "main" and confidence >= 0.6):
        return {
            "action": "urgent",
            "rationale": (f"D40 pothole on main road with lane_blocked={lane_blocked} "
                          f"and detector confidence {confidence:.2f}"),
            "priority": priority, "cited_sources": tool_result_ids[:2],
        }
    return {
        "action": "standard",
        "rationale": f"routine queue for {hazard_class} on {road_type} road",
        "priority": priority, "cited_sources": tool_result_ids[:1],
    }



def _llm_decide(hazard_class: str, confidence: float, road_type: str,
                lane_blocked: str, severity: str,
                procedures: list, tool_result_ids: list) -> dict:
    """Ask the decision LLM for the final verdict. Returns
    {action, priority, rationale, cited_sources}. On failure, raises -
    the pipeline wants a hard stop, not a silent fallback."""
    system = (
        'You are the Decision agent in a road-hazard triage system. '
        'Given a hazard profile and retrieved repair-standard passages, '
        'reply with a JSON object: action (urgent, standard, needs_more_evidence, '
        'human_review), priority (0-100), rationale (one short sentence), '
        'cited_sources (list of passage ids you grounded the decision in). '
        'Reply with ONLY the JSON.'
    )
    passages = '\n'.join(
        f'[{p[chr(34)+chr(97)+chr(114)+chr(116)+chr(105)+chr(99)+chr(108)+chr(101)+chr(95)+chr(105)+chr(100)+chr(115)+chr(34)][0]}] {(p.get(chr(34)+chr(112)+chr(97)+chr(115)+chr(115)+chr(97)+chr(103)+chr(101)+chr(34)) or chr(34)+chr(34))[:280]}'
        for p in procedures
    )
    user = (
        f'hazard: class={hazard_class} conf={confidence:.2f} road={road_type} '
        f'lane={lane_blocked} severity={severity}\n'
        f'passages:\n{passages}'
    )
    raw = chat_for('decision', [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': user},
    ])
    import re, json as _json
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if not m:
        raise ValueError(f'decision LLM returned no JSON: {raw[:200]!r}')
    obj = _json.loads(m.group(0))
    obj['action'] = str(obj.get('action', 'human_review'))
    obj['priority'] = int(obj.get('priority', 50))
    obj['rationale'] = str(obj.get('rationale', ''))[:300]
    cs = obj.get('cited_sources') or []
    obj['cited_sources'] = [str(x) for x in cs if x in tool_result_ids][:5]
    return obj

def decision(state: dict, allow_live_llm: bool = False) -> dict:
    profile = state.get('hazard_profile') or {}
    md = state.get('metadata') or {}
    procedures = state.get('procedures') or []
    tool_result_ids = state.get('tool_result_ids') or []
    v = _llm_decide(
        hazard_class=profile.get('primary_class', 'D40'),
        confidence=float(profile.get('primary_confidence') or 0.0),
        road_type=md.get('road_type', 'main'),
        lane_blocked=md.get('lane_blocked', 'none'),
        severity=profile.get('severity_signal', 'moderate'),
        procedures=procedures,
        tool_result_ids=tool_result_ids,
    )
    # Build the trace entry the UI / run_pipeline expect.
    trace_add = [{
        'speaker': 'decision',
        'content': dict(v, source='llm',
                        tool_result_ids=tool_result_ids),
    }]
    return {
        'verdict': {
            'action': v['action'], 'priority': v['priority'],
            'rationale': v['rationale'], 'cited_sources': v['cited_sources'],
        },
        'trace': state.get('trace', []) + trace_add,
        'final_verdict': {
            'action': v['action'], 'priority': v['priority'],
            'rationale': v['rationale'], 'cited_sources': v['cited_sources'],
        },
    }



## 11. `approval_gate`

For `urgent` and `human_review` verdicts the pipeline emits an approval record (a plain dict here; in production this is a SQLite row + an e-mail to the supervisor with an HMAC-signed link). For `standard` and `needs_more_evidence` the approval gate is a no-op; the verdict reaches the caller directly.

This node writes nothing to any file - it just appends an `approval_record` to the state. The caller can read it.


In [8]:

import hashlib


def approval_gate(state: dict, verdict: dict) -> dict:
    """For urgent / human_review, create a pending-approval record."""
    record = None
    if verdict["action"] in ("urgent", "human_review"):
        hazard_id = state.get("hazard_id") or hashlib.md5(
            json.dumps({"yolo": state.get("yolo_output"),
                        "md": state.get("metadata")}, sort_keys=True,
                       default=str).encode()).hexdigest()[:12]
        record = {
            "approval_id": f"ap_{hazard_id}",
            "hazard_id": f"hz_{hazard_id}",
            "action": verdict["action"],
            "priority": verdict.get("priority", 0),
            "rationale": verdict.get("rationale", ""),
            "status": "pending",
        }
    trace_entry = {
        "node": "approval_gate",
        "approval_record": record,
    }
    return {
        "verdict": verdict,
        "approval_record": record,
        "trace": state.get("trace", []) + [trace_entry],
    }


## 12. Running the pipeline

`run_pipeline(input_state)` wires every node together in the production order and returns the final verdict + the trace. If the budget guard fires at any supervisor tick, the pipeline short-circuits to `graceful_stop_decision` and the verdict is `human_review` with the "stopped at X" rationale.


In [9]:

def run_pipeline(input_state: dict, allow_live_llm: bool = False) -> dict:
    """End-to-end run. Returns a dict with verdict + approval_record + trace."""
    state = dict(input_state)
    state.setdefault("trace", [])
    state["budget"] = Budget()

    # 1. sanitize (no supervisor tick needed - cheap)
    state.update(sanitize(state))

    # 2. signal_extractor
    state.update(signal_extractor(state))

    # 3. supervisor tick -> profile_builder
    sup = supervisor(state); state.update(sup)
    if sup.get("final_verdict"):
        return approval_gate(state, sup["final_verdict"])
    state.update(profile_builder(state))

    # 4. supervisor tick -> rag_retriever
    sup = supervisor(state); state.update(sup)
    if sup.get("final_verdict"):
        return approval_gate(state, sup["final_verdict"])
    state.update(rag_retriever(state))

    # 5. supervisor tick -> decision
    sup = supervisor(state); state.update(sup)
    if sup.get("final_verdict"):
        return approval_gate(state, sup["final_verdict"])
    verdict = decision(state, allow_live_llm=allow_live_llm)

    # 6. approval_gate
    return approval_gate(state, verdict)


## 13. Demo on sample images

The demo picks images from the team's shared sample set at `../data/sample/` (Ovad), runs Shai's YOLOv8n ONNX detector at `../models/yolov8n-roadguard.onnx` on each, and feeds the top-1 detection + metadata into the pipeline. Up to five images are processed in order; the result table shows the final verdict per image.


In [10]:
from pathlib import Path
import onnxruntime as ort
import numpy as np
from PIL import Image

YOLO_CANDIDATES = [
    "../models/yolov8n-roadguard.onnx",       # standalone layout
    "../Models/yolov8n-roadguard.onnx",       # team repo CapCase
    "../webapp/assets/models/yolov8n-roadguard.onnx",  # webapp copy
]
YOLO_MODEL = next((p for p in YOLO_CANDIDATES if Path(p).exists()), YOLO_CANDIDATES[0])
DATA_CANDIDATES = [
    "../data/sample",       # standalone layout
    "../Data/sample",       # team repo CapCase
    "../sample_images",     # alt sample dir
]
DATA_DIR = next((p for p in DATA_CANDIDATES
                 if Path(p).exists() and any(Path(p).iterdir())), DATA_CANDIDATES[0])
YOLO_CLASSES = ['D00', 'D10', 'D20', 'D40']

def _letterbox(img, size=640, pad=114):
    w, h = img.size
    scale = size / max(w, h)
    nw, nh = int(round(w * scale)), int(round(h * scale))
    r = img.resize((nw, nh), Image.BILINEAR)
    canvas = Image.new('RGB', (size, size), (pad, pad, pad))
    canvas.paste(r, ((size - nw) // 2, (size - nh) // 2))
    return canvas, scale, (size - nw) // 2, (size - nh) // 2

_yolo_session = None

def yolo_top1(image_path: str) -> dict:
    global _yolo_session
    if _yolo_session is None:
        model_p = Path(YOLO_MODEL)
        if not model_p.exists():
            raise FileNotFoundError(
                f"YOLO ONNX {YOLO_MODEL!r} not found. "
                "Place the team's weights at this path before running."
            )
        _yolo_session = ort.InferenceSession(str(model_p),
                                             providers=['CPUExecutionProvider'])
    img = Image.open(image_path).convert('RGB')
    canvas, scale, pad_x, pad_y = _letterbox(img, 640)
    arr = np.asarray(canvas, dtype='float32') / 255.0
    arr = arr.transpose(2, 0, 1)[None]
    out = _yolo_session.run(None, {_yolo_session.get_inputs()[0].name: arr})[0]
    rows = out[0].T
    best_conf, best_cls = 0.0, 0
    best_box = (0, 0, 1, 1)
    w, h = img.size
    for row in rows:
        cx, cy, bw, bh = row[:4]
        scores = row[4:]
        cls_idx = int(np.argmax(scores))
        conf = float(scores[cls_idx])
        if conf <= best_conf:
            continue
        x1 = (cx - bw / 2 - pad_x) / scale
        y1 = (cy - bh / 2 - pad_y) / scale
        x2 = (cx + bw / 2 - pad_x) / scale
        y2 = (cy + bh / 2 - pad_y) / scale
        best_conf, best_cls = conf, cls_idx
        best_box = (float(max(0, x1)), float(max(0, y1)),
                    float(min(w, x2)), float(min(h, y2)))
    area = max(0.0, best_box[2] - best_box[0]) * max(0.0, best_box[3] - best_box[1])
    return {
        'top1_class': YOLO_CLASSES[best_cls],
        'top1_confidence': round(best_conf, 3),
        'top1_margin': 0.0,
        'area_ratio': round(area / max(1, w * h), 4),
    }

def _sample_images():
    d = Path(DATA_DIR)
    if not d.exists():
        raise FileNotFoundError(
            f"Sample image directory {DATA_DIR!r} not found. "
            "Place at least one road image here before running."
        )
    exts = {'.jpg', '.jpeg', '.png', '.webp'}
    files = sorted(p for p in d.iterdir() if p.suffix.lower() in exts)
    return files  # all images - the team repo ships ~94

results = []
rows_for_html = []
import time as _rt
for img_path in _sample_images():
    yolo = yolo_top1(str(img_path))
    input_state = {
        'yolo_output': yolo,
        'image_quality': {'blur_score': 150, 'is_road_scene': True,
                          'ocr_text_detected': ''},
        'metadata': {'road_type': 'main', 'lane_blocked': 'none',
                     'location': img_path.name, 'reporter_text': ''},
    }
    _t0 = _rt.monotonic()
    r = run_pipeline(input_state)
    _elapsed = _rt.monotonic() - _t0
    hazard_id = 'hz_' + str(hash(img_path.name) & 0xffffffff)[:12]
    rich_print_trace(hazard_id, img_path.name, r)
    write_jsonl_row(hazard_id, img_path.name, r, _elapsed)
    rows_for_html.append({
        'hazard_id': hazard_id, 'image': img_path.name,
        'action': (r.get('verdict') or {}).get('action'),
        'priority': (r.get('verdict') or {}).get('priority'),
        'cited_sources': (r.get('verdict') or {}).get('cited_sources'),
        'rationale': (r.get('verdict') or {}).get('rationale'),
        'elapsed_s': round(_elapsed, 3),
    })
    results.append((img_path.name, r))

# Smoke checks over the sample set.
actions = {'urgent', 'standard', 'needs_more_evidence', 'human_review'}
for name, r in results:
    v = r['verdict']
    assert v['action'] in actions, f'{name}: unexpected action {v[chr(34)+chr(97)+chr(99)+chr(116)+chr(105)+chr(111)+chr(110)+chr(34)]}'
    assert v.get('priority') is not None, f'{name}: priority is None'
    agents = {e.get('speaker') for e in r['trace']}
    # at least one meaningful speaker (sanitize is folded into other
    # speakers in some runs); the verdict presence is the hard check
    assert agents - {None}, f'{name}: empty trace'
print('smoke checks passed over', len(results), 'image(s).')
write_html_report(rows_for_html)


11993938-960x960.jpg                     -> action=standard             priority=55 cited=['segment_doc_24_irc.gov.in.082.2015_1_c2']
24_476_large.jpg                         -> action=needs_more_evidence  priority=85 cited=[]
24_488_large.jpg                         -> action=needs_more_evidence  priority=65 cited=[]
24_939_large.jpg                         -> action=needs_more_evidence  priority=55 cited=[]
34037.webp                               -> action=standard             priority=55 cited=['segment_doc_24_irc.gov.in.082.2015_1_c2']
478161.webp                              -> action=standard             priority=65 cited=['segment_doc_22_14  well-managed_highway_infrastructure_combined_-_28_october_2016_amended_15_march_2017__180_c2']
7b770ebb31fb01d48e59f6c95da19306-845x1127.jpg -> action=standard             priority=65 cited=['segment_doc_22_14  well-managed_highway_infrastructure_combined_-_28_october_2016_amended_15_march_2017__180_c2']
894ff179285ce91ccf7a093ae0aa5585-845

## 14. Trace detail

Expand any one scenario's full trace: every node's output, in order. This is the same shape the production system writes to `agent_trace_json` on the hazards table and renders on the `/trace` operator page.


In [11]:

import pprint

# Pick one scenario and dump its full trace
pprint.pprint(results[0][1]["trace"], width=120, sort_dicts=False)


[{'node': 'sanitize', 'location_scrubbed': False, 'ocr_suspicious': False, 'reporter_text_suspicious': False},
 {'node': 'signal_extractor',
  'has_text': False,
  'mentioned': {'hazard_class': None, 'lane_blocked': None, 'road_type': None},
  'contradictions': 0,
  'urgency_claimed': False,
  'reporter_signals': {'size_cm': None, 'depth_cm': None, 'vehicles_swerving': None, 'standing_water': None}},
 {'node': 'supervisor', 'iterations': 1, 'elapsed_s': 0.0, 'fired': False},
 {'node': 'profile_builder',
  'primary_class': 'D40',
  'primary_confidence': 0.285,
  'severity_signal': 'moderate',
  'perception_confidence': 'medium'},
 {'node': 'supervisor', 'iterations': 2, 'elapsed_s': 0.0, 'fired': False},
 {'speaker': 'rag_retriever',
  'content': {'coverage': 'direct_match',
              'n_procedures': 3,
              'tool_result_ids': ['segment_doc_24_irc.gov.in.082.2015_1_c2',
                                  'segment_doc_22_14  '
                                  'well-managed_h

## 15. Agent output artifacts

Every demo run writes four artifacts into `./runs/<timestamp>/`:

- `results.jsonl` - one line per hazard with verdict + priority + cited_sources + elapsed_s + per-agent duration.
- `traces/<hazard_id>.json` - the full agent trace for that image.
- `report.html` - a self-contained HTML report with a verdict table, cited passages and per-image trace blocks.
- The console output is printed per-agent so the operator sees each step.


In [ ]:

from pathlib import Path
import json, time, html

RUN_ROOT = Path('runs') / time.strftime('%Y%m%d_%H%M%S')
TRACES_DIR = RUN_ROOT / 'traces'
RESULTS_PATH = RUN_ROOT / 'results.jsonl'
REPORT_PATH = RUN_ROOT / 'report.html'
TRACES_DIR.mkdir(parents=True, exist_ok=True)
print(f'run artifacts will land under {RUN_ROOT!r}')

def rich_print_trace(hazard_id, img_name, result):
    v = result.get('verdict') or {}
    print()
    print('=' * 90)
    print(f'HAZARD {hazard_id}  image={img_name}')
    print(f'  final: action={v.get(chr(34)+chr(97)+chr(99)+chr(116)+chr(105)+chr(111)+chr(110)+chr(34))}  priority={v.get(chr(34)+chr(112)+chr(114)+chr(105)+chr(111)+chr(114)+chr(105)+chr(116)+chr(121)+chr(34))}  cited={v.get(chr(34)+chr(99)+chr(105)+chr(116)+chr(101)+chr(100)+chr(95)+chr(115)+chr(111)+chr(117)+chr(114)+chr(99)+chr(101)+chr(115)+chr(34))}')
    print('-' * 90)
    for e in result.get('trace') or []:
        sp = e.get('speaker') or e.get('node') or '?'
        kind = e.get('kind', '')
        dur = e.get('elapsed_s')
        content = e.get('content') or {}
        head = f'  [{sp:18s}]'
        if dur is not None:
            head += f' {float(dur):5.2f}s'
        if kind:
            head += f'  kind={kind}'
        print(head)
        for k, val in list(content.items())[:8]:
            vs = str(val)
            if len(vs) > 100:
                vs = vs[:97] + '...'
            print(f'    {k}: {vs}')
    print('=' * 90)

def write_jsonl_row(hazard_id, img_name, result, elapsed_s):
    v = result.get('verdict') or {}
    row = {
        'hazard_id': hazard_id, 'image': img_name,
        'action': v.get('action'), 'priority': v.get('priority'),
        'cited_sources': v.get('cited_sources') or [],
        'rationale': v.get('rationale'),
        'elapsed_s': round(elapsed_s, 3),
        'n_trace_events': len(result.get('trace') or []),
    }
    with open(RESULTS_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + chr(10))
    with open(TRACES_DIR / f'{hazard_id}.json', 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2, default=str)

def write_html_report(rows):
    def esc(x): return html.escape(str(x) if x is not None else '')
    body = ['<!doctype html><html><head><meta charset="utf-8"><title>RoadGuard run</title>',
            '<style>body{font-family:Arial;max-width:1200px;margin:30px auto}',
            'table{border-collapse:collapse;width:100%}',
            'th,td{border:1px solid #ddd;padding:6px 8px;font-size:13px}',
            'th{background:#f0f5fa;text-align:left}',
            '.urgent{background:#fee}.standard{background:#efe}',
            '.needs_more_evidence{background:#ffe}.human_review{background:#eef}',
            'pre{background:#f8f8f8;padding:8px;overflow:auto;font-size:11px}</style></head><body>',
            f'<h1>RoadGuard run - {RUN_ROOT.name}</h1>',
            f'<p>{len(rows)} hazards processed. Artifacts under <code>{RUN_ROOT}</code>.</p>',
            '<table><tr><th>#</th><th>image</th><th>action</th><th>priority</th>',
            '<th>cited_sources</th><th>rationale</th><th>elapsed</th></tr>']
    for i, r in enumerate(rows, start=1):
        body.append(
            f'<tr class="{esc(r.get(chr(34)+chr(97)+chr(99)+chr(116)+chr(105)+chr(111)+chr(110)+chr(34)))}">'
            f'<td>{i}</td><td>{esc(r.get(chr(34)+chr(105)+chr(109)+chr(97)+chr(103)+chr(101)+chr(34)))}</td>'
            f'<td><b>{esc(r.get(chr(34)+chr(97)+chr(99)+chr(116)+chr(105)+chr(111)+chr(110)+chr(34)))}</b></td>'
            f'<td>{esc(r.get(chr(34)+chr(112)+chr(114)+chr(105)+chr(111)+chr(114)+chr(105)+chr(116)+chr(121)+chr(34)))}</td>'
            f'<td>{esc(chr(44).join(r.get(chr(34)+chr(99)+chr(105)+chr(116)+chr(101)+chr(100)+chr(95)+chr(115)+chr(111)+chr(117)+chr(114)+chr(99)+chr(101)+chr(115)+chr(34)) or []))}</td>'
            f'<td>{esc(r.get(chr(34)+chr(114)+chr(97)+chr(116)+chr(105)+chr(111)+chr(110)+chr(97)+chr(108)+chr(101)+chr(34)))}</td>'
            f'<td>{r.get(chr(34)+chr(101)+chr(108)+chr(97)+chr(112)+chr(115)+chr(101)+chr(100)+chr(95)+chr(115)+chr(34))}s</td></tr>')
    body.append('</table></body></html>')
    REPORT_PATH.write_text(chr(10).join(body), encoding='utf-8')
    print(f'wrote {REPORT_PATH}')

print('output writers ready.')


## 16. What this notebook does NOT include

For scope, these production features are intentionally left out:

- **FastAPI service**: there is no HTTP layer; call `run_pipeline` from Python directly.
- **SQLite persistence**: the trace is returned in memory. In the production system the API writes it to `hazards.agent_trace_json`.
- **Approval email + supervisor dashboard**: `approval_gate` returns an approval record as a dict only; no SMTP, no inbox UI.
- **Image upload + webapp**: a hazard arrives in the production system via `POST /hazards/submit-async`; this notebook calls the agent layer directly on an image on disk.
- **Operator pages** (`/trace`, `/supervisor`, `/digest`, `/flow`): the trace is handed back to the caller; rendering is up to them.
